# 🤖 Interfaz Multi-IA — Week 2

Plantilla interactiva construida con **Gradio** que permite interactuar con múltiples proveedores de IA 
desde una única interfaz, soportando respuestas en texto, voz e imagen, y con acceso a herramientas 
externas mediante *tool calling*.

---

## 🧩 IAs y modelos soportados

| Proveedor | Modelos por defecto |
|-----------|-------------------|
| **OpenAI** | gpt-4o-mini, gpt-3.5-turbo, gpt-4o |
| **Anthropic** | claude-haiku-4-5, claude-sonnet-4-6, claude-3-5-haiku |
| **Groq / Llama** | llama-3.1-8b-instant, llama-3.3-70b-versatile, llama-4-scout |
| **Qwen** | qwen3-8b, qwen3-14b, qwen3-32b |
| **DeepSeek** | deepseek-chat, deepseek-reasoner, deepseek-coder |
| **Gemini** | gemini-2.0-flash-lite, gemini-2.0-flash, gemini-2.5-flash |

La lista de modelos puede actualizarse en tiempo real consultando la API de cada proveedor 
mediante el botón **Actualizar modelos IA**.

---

## 🛠️ Herramientas disponibles (Tool Calling)

Las siguientes herramientas están disponibles para OpenAI, Anthropic y Groq. 
El LLM las invoca automáticamente según el contexto del mensaje:

- **`get_datetime`** — Devuelve la fecha, hora y día de la semana actual del sistema.
- **`get_weather`** — Obtiene el clima actual de cualquier ciudad vía Open-Meteo (sin API key).
- **`get_flights`** — Consulta vuelos activos en tiempo real desde un aeropuerto vía AviationStack API. Requiere `AVIATIONSTACK_API_KEY`.

---

## 🖼️ Tipos de salida

- **Texto** — Respuesta en streaming directamente en la interfaz.
- **Voz** — Síntesis de voz mediante OpenAI TTS (modelo `tts-1`, voz `onyx`), reproducida con ffplay.
- **Imagen** — Generación de imagen a partir del mensaje, con tres motores disponibles:
  - OpenAI (`gpt-image-1`)
  - Hugging Face (`FLUX.1-schnell`)
  - Pollinations (gratuito, sin API key)

---

## 🔑 Variables de entorno requeridas (fichero `.env`)


## 📋 Estructura del código

1. **Librerías e imports**
2. **Constantes** — `MAX_TOKENS`, `MAX_WORDS`, `TEMPERATURE`
3. **Carga de API keys** y clientes de cada IA
4. **Herramientas** — `get_datetime`, `get_weather`, `get_flights`, `handle_tool_call`
5. **Funciones de llamada** — `call_openai`, `call_anthropic`, `call_groq`, `call_qwen`, `call_deepseek`, `call_gemini`
6. **Funciones de imagen** — Pollinations, Hugging Face, OpenAI
7. **Funciones auxiliares** — `call_modelo`, `validar_y_ejecutar`, `actualizar_modelos`, `obtener_modelos_api`
8. **Interfaz Gradio** — `gr.Blocks` con eventos dinámicos


## ⚠️ Observaciones y limitaciones

### Herramienta `get_flights`
- El LLM no siempre invoca la herramienta de forma automática. Para garantizar su uso es recomendable incluir un **system prompt explícito** indicando que debe consultar `get_flights` ante cualquier pregunta sobre vuelos.
- La API gratuita de AviationStack solo devuelve **vuelos activos en tiempo real**. No soporta búsquedas por destino, fechas futuras ni historial de vuelos.
- Si el usuario pregunta por vuelos a un destino concreto o en una fecha futura, la herramienta no puede responder esa consulta y el LLM tenderá a responder con conocimiento general.

### Tool Calling
- **DeepSeek** y **Gemini** no están integrados con tool calling en esta implementación. Solo funcionan en modo texto sin acceso a herramientas.
- Sin system prompt, algunos modelos tienen dificultades para decidir cuándo invocar una herramienta. Se recomienda siempre definir el rol del asistente en el campo "Prompt de Sistema".

### Modelos de imagen
- **OpenAI (`gpt-image-1`)** — Mayor calidad, pero tiene coste por imagen generada.
- **Hugging Face (`FLUX.1-schnell`)** — Gratuito con cuenta registrada, buena calidad, puede tener latencia según la carga del servicio.
- **Pollinations** — Totalmente gratuito y sin registro, pero con menor estabilidad y tiempos de respuesta variables.

### Streaming
- El streaming de la respuesta final solo está activo en OpenAI, Anthropic y Groq. DeepSeek y Gemini devuelven la respuesta completa de una vez.
- Cuando se activa tool calling, la primera llamada al LLM se realiza sin streaming para poder capturar correctamente la invocación de la herramienta.

In [1]:
# Librerías

from dotenv import load_dotenv                                     # Carga variables de entorno desde el fichero .env
from IPython.display import Markdown, display, update_display      # Muestra contenido Markdown en Jupyter
import os                                                          # Librería estándar de Python para interactuar con el SO
import requests                                                    # Librería HTTP de Python — permite hacer llamadas REST a servicios externos como la API de Ollama
import gradio as gr                                                # Framework para crear interfaces web interactivas para modelos de IA

from openai import OpenAI                                          # Cliente oficial de OpenAI para llamadas a la API
#import ollama                                                     # Cliente oficial de Ollama para modelos locales
from groq import Groq                                              # Cliente oficial de Groq para modelos LLM en cloud (Llama, Qwen)
import anthropic                                                   # Cliente oficial de Anthropic para llamadas a la API de Claude
import google.genai as genai                                       # Cliente oficial de Google para llamadas a la API de Gemini

from io import BytesIO                                             # Gestionar datos en memoria como si fuera un fichero.Requerido para gestionar el audio
import tempfile                                                    # Permite crear ficheros temporales en el sistema. Gradio necesita una ruta de fichero para reproducir el audio
import subprocess                                                  # Librería estándar de Python que permite ejecutar comandos del sistema operativo desde Python

import urllib.parse                                                # Librería imagen Pollinations
from PIL import Image
import base64

from datetime import datetime
import json

In [2]:
# Control de respuesta
MAX_TOKENS  = 1200            # Límite máximo de tokens en la respuesta. Riesgo: Respuesta incompleta. Corta respuesta al alcanzar el número de tokens.
MAX_WORDS   = 300             # Límite orientativo de palabras indicado en el system prompt
TEMPERATURE = 0.7

In [3]:
# Cargar variables de entorno en un archivo llamado .env

load_dotenv()

API_KEYS = {
    "openai":    os.getenv("OPENAI_API_KEY"),
    "anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "groq":      os.getenv("GROQ_API_KEY"),
    "deepseek":  os.getenv("DEEPSEEK_API_KEY"),
    "gemini":    os.getenv("GOOGLE_API_KEY"),
    "huggingface": os.getenv("HUGGINGFACE_TOKEN"),
    "aviationstack": os.getenv("AVIATIONSTACK_API_KEY")
}

In [4]:
#Verificar API KEY disponibles
for nombre, clave in API_KEYS.items():
    if clave:
        print(f"✓ {nombre}: {clave[:8]}...")
    else:
        print(f"✗ {nombre}: No configurada")

✓ openai: sk-proj-...
✓ anthropic: sk-ant-a...
✓ groq: gsk_5cF1...
✓ deepseek: sk-6d10b...
✓ gemini: AQ.Ab8RN...
✓ huggingface: hf_UQyqM...
✓ aviationstack: de9bf91a...


In [5]:
# Lista de modelos implementados

MODELOS = {
    "OpenAI":     ["gpt-4o-mini", "gpt-3.5-turbo", "gpt-4o"],
    "Anthropic":  ["claude-haiku-4-5-20251001", "claude-sonnet-4-6", "claude-3-5-haiku-20250514"],
    "Groq/Llama": ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", "meta-llama/llama-4-scout-17b-16e-instruct"],
    "Qwen":       ["qwen/qwen3-32b", "qwen/qwen3-8b", "qwen/qwen3-14b"],
    "DeepSeek":   ["deepseek-chat", "deepseek-reasoner", "deepseek-coder"],
    "Gemini":     ["gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash"]
}

MODELOS_IMAGEN = ["OpenAI (gpt-image-1)", "Hugging Face (FLUX.1-schnell)", "Pollinations (gratuito)"]

In [6]:
# Clientes de cada IA

clients = {}

try:
    clients["OpenAI"] = OpenAI(api_key=API_KEYS["openai"])
    print("✓ OpenAI inicializado")
except Exception as e:
    print(f"✗ OpenAI ERROR: {e}")

try:
    clients["Anthropic"] = anthropic.Anthropic(api_key=API_KEYS["anthropic"])
    print("✓ Anthropic inicializado")
except Exception as e:
    print(f"✗ Anthropic ERROR: {e}")

try:
    clients["Groq/Llama"] = Groq(api_key=API_KEYS["groq"])
    clients["Qwen"] = Groq(api_key=API_KEYS["groq"])
    print("✓ Groq/Llama y Qwen inicializados")
except Exception as e:
    print(f"✗ Groq ERROR: {e}")

try:
    clients["DeepSeek"] = OpenAI(api_key=API_KEYS["deepseek"], base_url="https://api.deepseek.com")
    print("✓ DeepSeek inicializado")
except Exception as e:
    print(f"✗ DeepSeek ERROR: {e}")

try:
    clients["Gemini"] = genai.Client(api_key=API_KEYS["gemini"])
    print("✓ Gemini inicializado")
except Exception as e:
    print(f"✗ Gemini ERROR: {e}")

✓ OpenAI inicializado
✓ Anthropic inicializado
✓ Groq/Llama y Qwen inicializados
✓ DeepSeek inicializado
✓ Gemini inicializado


In [7]:
# Herramienta - Fecha y hora

def get_datetime():
    now = datetime.now()
    return {
        "fecha": now.strftime("%d/%m/%Y"),
        "hora": now.strftime("%H:%M:%S"),
        "dia_semana": now.strftime("%A")
    }

In [8]:
# Herramienta - Clima

def get_weather(ciudad):
    
    # Primero obtenemos coordenadas de la ciudad
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={ciudad}&count=1&language=es"
    geo_response = requests.get(geo_url).json()
    
    if not geo_response.get("results"):
        return {"error": f"Ciudad '{ciudad}' no encontrada"}
    
    lat = geo_response["results"][0]["latitude"]
    lon = geo_response["results"][0]["longitude"]
    
    # Luego obtenemos el clima
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,wind_speed_10m,precipitation,weather_code"
    weather_response = requests.get(weather_url).json()
    current = weather_response["current"]
    
    return {
        "ciudad": ciudad,
        "temperatura": f"{current['temperature_2m']}°C",
        "viento": f"{current['wind_speed_10m']} km/h",
        "precipitacion": f"{current['precipitation']} mm"
    }

In [9]:
# Herramienta - Busca vuelos

def get_flights(ciudad):
    """Busca vuelos de salida desde un aeropuerto dado su código IATA o nombre de ciudad."""
    url = "http://api.aviationstack.com/v1/flights"
    params = {
        "access_key": os.environ.get("AVIATIONSTACK_API_KEY"),
        "dep_iata": ciudad.upper(),
        "flight_status": "active",
        "limit": 10
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if not data.get("data"):
            return f"No se encontraron vuelos activos saliendo de {ciudad.upper()}."
        
        vuelos = []
        for vuelo in data["data"]:
            numero = vuelo.get("flight", {}).get("iata", "N/A")
            aerolinea = vuelo.get("airline", {}).get("name", "N/A")
            destino = vuelo.get("arrival", {}).get("airport", "N/A")
            destino_iata = vuelo.get("arrival", {}).get("iata", "N/A")
            salida = vuelo.get("departure", {}).get("scheduled", "N/A")
            estado = vuelo.get("flight_status", "N/A")
            
            vuelos.append(
                f"✈️ {numero} | {aerolinea} → {destino} ({destino_iata}) | "
                f"Salida: {salida} | Estado: {estado}"
            )
        
        return "\n".join(vuelos)
    
    except requests.exceptions.RequestException as e:
        return f"Error al consultar la API de vuelos: {str(e)}"

resultado = get_flights("MAD")
print(resultado)

In [10]:
# Estructura de la tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_datetime",
            "description": "Obtiene la fecha y hora actual del sistema. Úsala cuando el usuario pregunte por la fecha, hora o día actual.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtiene el clima actual de una ciudad. Úsala cuando el usuario pregunte por el tiempo, temperatura o clima de una ciudad.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ciudad": {
                        "type": "string",
                        "description": "El nombre de la ciudad de la que se quiere conocer el clima"
                    }
                },
                "required": ["ciudad"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flights",
            "description": "SIEMPRE usa esta herramienta cuando el usuario mencione vuelos, aeropuertos, salidas, llegadas, aerolíneas o viajes en avión. Traduce automáticamente el nombre de la ciudad a su código IATA: Madrid=MAD, Barcelona=BCN, Valencia=VLC, Sevilla=SVQ, Bilbao=BIO, Lisboa=LIS, París=CDG, Londres=LHR, Ámsterdam=AMS, Roma=FCO.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ciudad": {
                        "type": "string",
                        "description": "Código IATA del aeropuerto de salida, ej: MAD, BCN, VLC"
                    }
                },
                "required": ["ciudad"]
            }
        }
    }
]


In [11]:
TOOLS_MAP = {
    "get_datetime": get_datetime,
    "get_weather": get_weather,
    "get_flights": get_flights
}

In [12]:
# Función que ejecuta la herramienta cuando el LLM la solicita

def handle_tool_call(tool_call):
    tool_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    resultado = TOOLS_MAP[tool_name](**arguments)
    
    return {
        "role": "tool",
        "content": json.dumps(resultado),
        "tool_call_id": tool_call.id
    }, f"🔧 Consultando {tool_name}..."

In [13]:
def call_openai(sistema, mensaje, ejemplo, modelo):

    print(f">>> MENSAJE RECIBIDO: {mensaje}")
    print(f">>> SISTEMA: {sistema}")
    
    messages = []
    if sistema:
        messages.append({"role": "system", "content": sistema})
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    messages.append({"role": "user", "content": mensaje})
    
    response = clients["OpenAI"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        tools=tools,
        stream=False
    )
    
    # Si el LLM quiere usar una herramienta
    while response.choices[0].finish_reason == "tool_calls":
        tool_call = response.choices[0].message.tool_calls[0]
        tool_response, info_mensaje = handle_tool_call(tool_call)
        yield info_mensaje
        
        # Añadir respuesta del LLM y resultado de la herramienta al historial
        messages.append(response.choices[0].message)
        messages.append(tool_response)
        
        # Nueva llamada al LLM con el resultado de la herramienta
        response = clients["OpenAI"].chat.completions.create(
            model=modelo,
            messages=messages,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            tools=tools,
            stream=False
        )

    # Fuera del while — respuesta final con stream
    stream_final = clients["OpenAI"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=True  # ← con stream solo al final
    )

    
    # Streaming de la respuesta final
    result = ""
    for chunk in stream_final:
        result += chunk.choices[0].delta.content or ""
        yield result

In [14]:
def call_anthropic(sistema, mensaje, ejemplo, modelo):
    messages = []
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    messages.append({"role": "user", "content": mensaje})
    
    response = clients["Anthropic"].messages.create(
        model=modelo,
        max_tokens=MAX_TOKENS,
        system=sistema if sistema else "Eres un asistente útil.",
        messages=messages,
        temperature=TEMPERATURE,
        tools=[{
            "name": t["function"]["name"],
            "description": t["function"]["description"],
            "input_schema": t["function"]["parameters"]
        } for t in tools]
    )
    
    # Si el LLM quiere usar una herramienta
    while response.stop_reason == "tool_use":
        tool_use = next(b for b in response.content if b.type == "tool_use")
        tool_call_obj = type('obj', (object,), {
            'function': type('obj', (object,), {
                'name': tool_use.name,
                'arguments': json.dumps(tool_use.input)
            })(),
            'id': tool_use.id
        })()
        
        tool_response, info_mensaje = handle_tool_call(tool_call_obj)
        yield info_mensaje
        
        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": tool_response["content"]
            }]
        })
        
        response = clients["Anthropic"].messages.create(
            model=modelo,
            max_tokens=MAX_TOKENS,
            system=sistema if sistema else "Eres un asistente útil.",
            messages=messages,
            temperature=TEMPERATURE
        )
    
    # Respuesta final con streaming
    with clients["Anthropic"].messages.stream(
        model=modelo,
        max_tokens=MAX_TOKENS,
        system=sistema if sistema else "Eres un asistente útil.",
        messages=messages
    ) as stream:
        result = ""
        for text in stream.text_stream:
            result += text
            yield result

In [15]:
def call_groq(sistema, mensaje, ejemplo, modelo):
    messages = []
    if sistema:
        messages.append({"role": "system", "content": sistema})
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    messages.append({"role": "user", "content": mensaje})
    
    response = clients["Groq/Llama"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        tools=tools,
        stream=False
    )
    
    while response.choices[0].finish_reason == "tool_calls":
        tool_call = response.choices[0].message.tool_calls[0]
        tool_response, info_mensaje = handle_tool_call(tool_call)
        yield info_mensaje
        messages.append(response.choices[0].message)
        messages.append(tool_response)
        response = clients["Groq/Llama"].chat.completions.create(
            model=modelo,
            messages=messages,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            tools=tools,
            stream=False
        )
    
    stream_final = clients["Groq/Llama"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=True
    )
    result = ""
    for chunk in stream_final:
        result += chunk.choices[0].delta.content or ""
        yield result

In [16]:
def call_qwen(sistema, mensaje, ejemplo, modelo):
    messages = []
    if sistema:
        messages.append({"role": "system", "content": sistema})
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    messages.append({"role": "user", "content": mensaje})
    
    response = clients["Qwen"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        tools=tools,
        stream=False
    )
    
    while response.choices[0].finish_reason == "tool_calls":
        tool_call = response.choices[0].message.tool_calls[0]
        tool_response, info_mensaje = handle_tool_call(tool_call)
        yield info_mensaje
        messages.append(response.choices[0].message)
        messages.append(tool_response)
        response = clients["Qwen"].chat.completions.create(
            model=modelo,
            messages=messages,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            tools=tools,
            stream=False
        )
    
    stream_final = clients["Qwen"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=True
    )
    result = ""
    for chunk in stream_final:
        result += chunk.choices[0].delta.content or ""
        yield result

In [17]:
def call_deepseek(sistema, mensaje, ejemplo, modelo):
    messages = []
    if sistema:
        messages.append({"role": "system", "content": sistema})
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    messages.append({"role": "user", "content": mensaje})
    
    stream = clients["DeepSeek"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=True
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [18]:
def call_gemini(sistema, mensaje, ejemplo, modelo):
    contents = ""
    if ejemplo:
        contents += f"Ejemplo:\n{ejemplo}\n\n"
    contents += mensaje
    
    response = clients["Gemini"].models.generate_content(
        model=modelo,
        config={
            "system_instruction": sistema if sistema else "Eres un asistente útil.",
            "max_output_tokens": MAX_TOKENS,
            "temperature": TEMPERATURE
        },
        contents=contents
    )
    return response.text

In [19]:
# Función para actualizar los modelos disponibles en función de la IA seleccionada

def actualizar_modelos(ia):
    modelos = MODELOS.get(ia, [])
    return gr.Dropdown(choices=modelos, value=modelos[0] if modelos else None)

In [20]:
def call_modelo(sistema, mensaje, ejemplo, ia, modelo):
    if ia == "OpenAI":
        yield from call_openai(sistema, mensaje, ejemplo, modelo)
    elif ia == "Anthropic":
        yield from call_anthropic(sistema, mensaje, ejemplo, modelo)
    elif ia == "Groq/Llama":
        yield from call_groq(sistema, mensaje, ejemplo, modelo)
    elif ia == "Qwen":
        yield from call_qwen(sistema, mensaje, ejemplo, modelo)
    elif ia == "DeepSeek":
        yield from call_deepseek(sistema, mensaje, ejemplo, modelo)
    elif ia == "Gemini":
        yield call_gemini(sistema, mensaje, ejemplo, modelo)
    else:
        yield f"⚠️ IA no reconocida: {ia}"

In [21]:
# Función para consultar vía API los modelos de IA y actualizar la lista 

def obtener_modelos_api(ia):

    #print(f"IA recibida: {ia}")
    
    try:
        if ia == "OpenAI":
            modelos = clients[ia].models.list()
            excluir = ["image", "transcribe", "tts", "instruct", "search", "codex"]
            lista = sorted([
                m.id for m in modelos.data
                if "gpt" in m.id
                and not any(x in m.id for x in excluir)
            ])
        
        elif ia == "Anthropic":
            modelos = clients[ia].models.list()
            lista = sorted([m.id for m in modelos.data])
        
        elif ia == "Groq/Llama":
            modelos = clients[ia].models.list()
            excluir = ["whisper", "orpheus", "prompt-guard", "safeguard", "compound"]
            lista = sorted([
                m.id for m in modelos.data
                if not any(x in m.id for x in excluir)
            ])
        
        elif ia == "Qwen":
            modelos = clients[ia].models.list()
            lista = sorted([m.id for m in modelos.data if "qwen" in m.id.lower()])
        
        elif ia == "DeepSeek":
            modelos = clients[ia].models.list()
            lista = sorted([m.id for m in modelos.data])
        
        elif ia == "Gemini":
            lista = [m.name for m in clients[ia].models.list()
                     if "gemini" in m.name.lower()]
        
        else:
            return MODELOS.get(ia, [])
        
        if lista:
            MODELOS[ia] = lista  # ← actualiza el diccionario global
            print(f"✓ {ia}: {len(lista)} modelos actualizados")
        else:
            print(f"⚠️ {ia}: No se obtuvieron modelos — se mantiene la lista actual")

        mensaje = f"✓ {ia}: {len(MODELOS[ia])} modelos disponibles"
        return gr.Dropdown(choices=MODELOS[ia], value=MODELOS[ia][0] if MODELOS[ia] else None), mensaje
    
    except Exception as e:
        print(f"⚠️ Error obteniendo modelos de {ia}: {e}")
        return gr.Dropdown(choices=MODELOS.get(ia, []), value=None), f"⚠️ Error: {str(e)}"
   

obtener_modelos_api("OpenAI")

obtener_modelos_api("Anthropic")

obtener_modelos_api("Groq/Llama")

obtener_modelos_api("Qwen")

obtener_modelos_api("DeepSeek")

obtener_modelos_api("Gemini")

In [22]:
# Función para reproducir Voz

def salida_audio(mensaje):
    response = clients["OpenAI"].audio.speech.create(
        model="tts-1",
        voice="onyx",
        input=mensaje
    )
    
    # Reproducir directamente desde memoria sin tocar disco
    ffplay_path = r"C:\Users\Jose M. Rivas\Developer\ffmpeg\bin\ffplay.exe"
    
    process = subprocess.Popen(
        [ffplay_path, "-nodisp", "-autoexit", "-hide_banner", "-i", "pipe:0"],
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    process.communicate(input=response.content)

In [23]:
#ruta = salida_audio("Qué te gustaría hacer en tus próximas vacaciones")

In [24]:
# Generar imagen mediante Pollinations - sin APIkey

def generar_imagen_pollinations(prompt):
    prompt_encoded = urllib.parse.quote(prompt)
    seed = int(datetime.now().timestamp())
    url = f"https://image.pollinations.ai/prompt/{prompt_encoded}?nologo=true&seed={seed}"
    response = requests.get(url, timeout=120)
    
    if response.status_code != 200:
        raise Exception(f"Pollinations no disponible (error {response.status_code}). Usa Hugging Face.")
    
    imagen = Image.open(BytesIO(response.content))
    return imagen

img = generar_imagen_pollinations("Una playa tropical al atardecer con palmeras") 
display(img)

In [25]:
def generar_imagen_huggingface(prompt):
    API_URL = "https://router.huggingface.co/hf-inference/models/black-forest-labs/FLUX.1-schnell"
    headers = {"Authorization": f"Bearer {API_KEYS['huggingface']}"}
    
    response = requests.post(API_URL, headers=headers, json={"inputs": prompt})
    imagen = Image.open(BytesIO(response.content))
    return imagen

img = generar_imagen_huggingface("Una playa tropical al atardecer con palmeras")
display(img)

In [26]:
def generar_imagen_openai(prompt):
    response = clients["OpenAI"].images.generate(
        model="gpt-image-1",
        prompt=prompt,
        size="1024x1024",
        n=1
    )
    image_base64 = response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    imagen = Image.open(BytesIO(image_data))
    return imagen

img = generar_imagen_openai("Una playa tropical al atardecer con palmeras")
display(img)

In [27]:
# Prepara el prompt para generar la imagen

def preparar_prompt_imagen(sistema, mensaje, ejemplo, ia, modelo):
    
    # Con contexto — usa system + ejemplo + mensaje sin enriquecer
    if sistema or ejemplo:
        messages = []
        if sistema:
            messages.append({"role": "system", "content": sistema})
        if ejemplo:
            messages.append({"role": "user", "content": "Ejemplo:"})
            messages.append({"role": "assistant", "content": ejemplo})
        messages.append({"role": "user", "content": 
           f"Usando el contexto anterior, genera un prompt detallado para un generador de imágenes. "
           f"Responde SOLO con el prompt.\n\n{mensaje}"
       })
        
        try:
            completion = clients[ia].chat.completions.create(
                model=modelo,
                messages=messages,
                max_tokens=300
            )
            return completion.choices[0].message.content
        except Exception:
            return mensaje
    
    # Solo mensaje — enriquecer con LLM
    else:
        prompt_instruccion = "Genera un prompt detallado y creativo para un generador de imágenes. Incluye estilo artístico, iluminación, colores y detalles visuales. Responde SOLO con el prompt, sin explicaciones."
        messages = [{"role": "user", "content": f"{prompt_instruccion}\n\n{mensaje}"}]
        
        try:
            completion = clients[ia].chat.completions.create(
                model=modelo,
                messages=messages,
                max_tokens=300
            )
            return completion.choices[0].message.content
        except Exception:
            return mensaje

In [28]:
# Verifica entrada de datos antes de llamar al modelo con los parámetros

def validar_y_ejecutar(sistema, mensaje, ejemplo, ia, modelo, tipo_salida, modelo_imagen):
   
    if not mensaje or not ia or not modelo:
        yield "⚠️ Por favor completa: Mensaje, IA y Modelo antes de continuar.", None
        return
    try:
        if tipo_salida == "Voz":
            yield "⏳ Generando respuesta...", None
            texto = ""
            for chunk in call_modelo(sistema, mensaje, ejemplo, ia, modelo):
                texto = chunk
            yield "🔊 Reproduciendo audio...", None
            salida_audio(texto)
            yield f"🔊 {texto}", None
        
        elif tipo_salida == "Imagen":
            yield "⏳ Preparando prompt de imagen...", None
            try:
                prompt = preparar_prompt_imagen(sistema, mensaje, ejemplo, ia, modelo)
            except Exception:
                prompt = mensaje
            yield f"⏳ Generando imagen con {modelo_imagen}...\n\nPrompt: {prompt}", None
    
            try:
                if modelo_imagen == "OpenAI (gpt-image-1)":
                    imagen = generar_imagen_openai(prompt)
                elif modelo_imagen == "Hugging Face (FLUX.1-schnell)":
                    imagen = generar_imagen_huggingface(prompt)
                else:
                    imagen = generar_imagen_pollinations(prompt)
                yield f"📝 Prompt:\n{prompt}", imagen
            except Exception as e:
                yield f"⚠️ Error generando imagen: {str(e)}", None
        
        else:
            for chunk in call_modelo(sistema, mensaje, ejemplo, ia, modelo):
                yield chunk, None
    
    except Exception as e:
        yield f"⚠️ Error: {str(e)}", None

In [29]:
# Actualización dinámica de controles en función del tipo de salida

def actualizar_controles(tipo_salida):
    desactivar = tipo_salida == "Voz"
    es_imagen = tipo_salida == "Imagen"
    return (
        gr.Dropdown(interactive=not desactivar),
        gr.Dropdown(interactive=not desactivar),
        gr.update(visible=es_imagen),
        gr.Textbox(visible=True),  # siempre visible para mensajes de estado
        gr.Image(visible=es_imagen)
    )

In [30]:
with gr.Blocks() as view:
    gr.Markdown("# 🤖 Interfaz Multi-IA")
    
    with gr.Row():
        with gr.Column():
            sistema  = gr.Textbox(label="Prompt de Sistema:", lines=3,
                                  placeholder="Define el rol del asistente...")
            ejemplo  = gr.Textbox(label="Prompt de Ejemplo (one-shot):", lines=3,
                                  placeholder="Ejemplo de pregunta/respuesta...")
            mensaje  = gr.Textbox(label="Tu Mensaje: *", lines=4,
                                  placeholder="Escribe tu mensaje aquí...")
            
            boton = gr.Button("Enviar", variant="primary")
            boton_clear = gr.Button("Limpiar", variant="secondary")
            boton_actualiza = gr.Button("Actualizar modelos IA", variant="secondary")
        
        with gr.Column():
            ia       = gr.Dropdown(list(MODELOS.keys()), label="Selecciona IA: *", value="OpenAI")
            modelo   = gr.Dropdown(MODELOS["OpenAI"], label="Selecciona Modelo: *", value=MODELOS["OpenAI"][0])
            tipo_salida  = gr.Radio(["Texto", "Voz", "Imagen"], label="Tipo de salida: *", value="Texto")
            modelo_imagen = gr.Dropdown(MODELOS_IMAGEN, label="Modelo de imagen:", value=MODELOS_IMAGEN[0], 
                                        visible=False)
            respuesta    = gr.Textbox(label="Respuesta:", lines=8, visible=True)
            salida_imagen = gr.Image(label="Imagen:", visible=False)
    
    #Eventos dinámicos
    tipo_salida.change(fn=actualizar_controles, 
                       inputs=tipo_salida, 
                       outputs=[ia, modelo, modelo_imagen, respuesta, salida_imagen])
    
    ia.change(fn=actualizar_modelos, inputs=ia, outputs=modelo)
  
    #boton = gr.Button("Enviar", variant="primary")
    boton.click(fn=validar_y_ejecutar,
                inputs=[sistema, mensaje, ejemplo, ia, modelo, tipo_salida, modelo_imagen], 
                outputs=[respuesta, salida_imagen])
    
    #boton_clear = gr.Button("Limpiar", variant="secondary")
    boton_clear.click(
        fn=lambda: ("", "", "", "", None),
        inputs=[],
        outputs=[sistema, ejemplo, mensaje, respuesta, salida_imagen]
    ) 
    
    #boton_actualiza = gr.Button("Actualizar modelos IA", variant="secondary")
    boton_actualiza.click(
        fn = obtener_modelos_api, 
        inputs=[ia], 
        outputs=[modelo, respuesta]
    )
    

view.launch(inbrowser=True, theme=gr.themes.Glass())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


⚠️ Error obteniendo modelos de Qwen: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}
⚠️ Error obteniendo modelos de Groq/Llama: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}
>>> MENSAJE RECIBIDO: Cuéntame un chiste
>>> SISTEMA: 
✓ DeepSeek: 2 modelos actualizados


for ia in MODELOS:
    print(f"\n=== {ia} ===")
    for m in MODELOS[ia]:
        print(f"  {m}")